# Frozen paired source-profile whole-group uncertainty


Predeclared: all152 genes/four unsupported/145 recorded groups; 500 whole-group draws, seed20261008, 2.5/97.5 percentiles with linear interpolation. Existing point estimates remain unchanged. Conditional on supplied groups; source/biological independence and per-gene calibration remain unresolved. No fitting, tuning or model selection.

In [1]:
from pathlib import Path
import hashlib,json,numpy as np
from starplast import group_ratios as G,functional_agreement as A
from starplast.scorecard_view import export_scorecard,render_scorecard_html
def sha(path):
 with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
source=ROOT/'starplast/data/functional_agreement.json'
assert sha(source)==A.REPORT_SHA256
report=json.loads(source.read_text())
A.build_scorecard(report)
paths=[source,ROOT/'starplast/group_ratios.py',ROOT/'tests/test_group_ratios.py',ROOT/'starplast/functional_agreement.py',ROOT/'starplast/scorecard_view.py',ROOT/'scripts/notebook_runner.py']
inputs={str(path):sha(path) for path in paths}
(OUT/'input_manifest.json').write_text(json.dumps(inputs,indent=2)+'\n')
state,arrays=G.estimate(*G.paired_terms(report),seed=20261008,draws=500)
assert state['entities']==152 and state['recorded_groups']==145
assert report['counts']['unsupported']==4
np.savez_compressed(OUT/'draws.npz',**arrays)
reloaded=np.load(OUT/'draws.npz')
replay,replay_arrays=G.estimate(*G.paired_terms(report),seed=20261008,draws=500,selections=reloaded['selections'])
assert replay==state
for name in arrays:np.testing.assert_array_equal(arrays[name],reloaded[name]);np.testing.assert_array_equal(arrays[name],replay_arrays[name])
view=G.paired_scorecard(report,state)
(OUT/'uncertainty.json').write_text(json.dumps(state,sort_keys=True,indent=2,allow_nan=False)+'\n')
(OUT/'scorecard.json').write_text(export_scorecard(view))
(OUT/'scorecard.html').write_text(render_scorecard_html(view,expanded=True))
assert json.loads((OUT/'scorecard.json').read_text())['snapshot']['group_uncertainty']==state
assert all(sha(path)==digest for path,digest in inputs.items())
{'groups':state['recorded_groups'],'draws':state['draws_evaluated'],'rates':state['rates']}

{'groups': 145, 'draws': 500, 'rates': {'joint_coverage': {'numerator': 143, 'denominator': 152, 'value': 0.9407894736842105, 'valid_draws': 500, 'undefined_draws': 0, 'interval': [0.9013157894736842, 0.9742803970223325], 'status': 'descriptive_group_bootstrap'}, 'agreement_among_joint_calls': {'numerator': 94, 'denominator': 143, 'value': 0.6573426573426573, 'valid_draws': 500, 'undefined_draws': 0, 'interval': [0.5771816707218167, 0.7370651236342726], 'status': 'descriptive_group_bootstrap'}, 'source_recovery_among_agreements': {'numerator': 33, 'denominator': 94, 'value': 0.35106382978723405, 'valid_draws': 500, 'undefined_draws': 0, 'interval': [0.25, 0.45809487951807226], 'status': 'descriptive_group_bootstrap'}, 'correct_agreements_all_eligible': {'numerator': 33, 'denominator': 152, 'value': 0.21710526315789475, 'valid_draws': 500, 'undefined_draws': 0, 'interval': [0.14851133391455973, 0.2894736842105263], 'status': 'descriptive_group_bootstrap'}, 'shared_wrong_among_joint_call